# Aula 04 — APIs, Bancos Vetoriais e Ferramentas Externas

**Disciplina:** Deploy de Agentes com LLMs — Pós-Graduação em Machine Learning (FIAP/POSTECH)

Damos ao agente **memória** (RAG sobre busca vetorial) e **ação** (tool use / function calling). Tudo com componentes **reais**:

1. **Embeddings** e busca semântica — `sentence-transformers` (Sentence-BERT)
2. **Índice vetorial** — FAISS
3. **RAG mínimo** — recuperar → fundamentar → gerar com **Claude**
4. **Tool use** — *function calling* nativo do Claude chamando uma **API externa** ao vivo

> Requer `ANTHROPIC_API_KEY` no arquivo `.env` (LLM real) e as libs do `requirements.txt`
> (`sentence-transformers`, `faiss-cpu`, `requests`). O passo 4 faz uma chamada HTTP real.

## Setup — Claude + modelo de embeddings

Carregamos a chave do `.env`, instanciamos o **Claude** (`claude-sonnet-4-6`) e o modelo de
embeddings **Sentence-BERT** (`all-MiniLM-L6-v2`). Ambos são reutilizados nas células seguintes.

In [ ]:
import numpy as np
from dotenv import load_dotenv, find_dotenv
from langchain_anthropic import ChatAnthropic
from sentence_transformers import SentenceTransformer

load_dotenv(find_dotenv(usecwd=True))  # lê aula-04/.env (ANTHROPIC_API_KEY=...)

llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def embed(textos):
    """Lista de textos -> vetores normalizados (produto interno = similaridade do cosseno)."""
    return np.asarray(embedder.encode(textos, normalize_embeddings=True), dtype="float32")

print("Claude e embeddings prontos. Dimensão dos vetores:", embed(["teste"]).shape[1])

## 1. Embeddings e busca semântica

Um **embedding** transforma texto em um vetor de significado; textos parecidos ficam próximos
no espaço. Como o modelo real captura **sinônimos**, *felino* casa com *gato* mesmo sem repetir
a palavra — a busca é por **significado**, não por correspondência exata.

In [ ]:
base = [
    "O gato dorme no sofá da sala",
    "O felino tira uma soneca no sofá",   # sentido próximo do 1º, sem repetir 'gato'
    "A bolsa de valores subiu hoje",
]
v = embed(base)
print("sim(gato, felino) =", round(float(v[0] @ v[1]), 3))
print("sim(gato, bolsa)  =", round(float(v[0] @ v[2]), 3))

consulta = "onde o bichano está descansando?"
scores = embed([consulta])[0] @ v.T
print(f"\nConsulta: {consulta!r}")
print("Frase mais similar:", base[int(np.argmax(scores))])

## 2. Índice vetorial com FAISS

Comparar um a um é caro em escala. Um **índice vetorial** recupera os `k` itens mais próximos
rapidamente. Usamos o FAISS `IndexFlatIP` (produto interno sobre vetores normalizados = cosseno).

In [ ]:
import faiss

catalogo = [
    "Teclado mecânico sem fio com switches silenciosos",
    "Mouse ergonômico para uso prolongado",
    "Monitor 27 polegadas 4K para design gráfico",
    "Cadeira de escritório com apoio lombar",
    "Webcam full HD com microfone embutido",
]
vetores = embed(catalogo)
index = faiss.IndexFlatIP(vetores.shape[1])
index.add(vetores)

consulta = "algo confortável para trabalhar muitas horas sentado"
dist, ids = index.search(embed([consulta]), k=3)
print(f"Consulta: {consulta!r}\n\nTop-3 mais relevantes:")
for pos, (i, d) in enumerate(zip(ids[0], dist[0]), start=1):
    print(f"  {pos}. (score {float(d):+.3f}) {catalogo[int(i)]}")

## 3. RAG mínimo — recuperar → fundamentar → gerar

RAG em 3 estágios: **armazenar** (chunks → embeddings → índice), **recuperar** (top-K mais
similares à pergunta) e **gerar** com o Claude, instruído a responder **somente** com base no
contexto recuperado. Assim o agente acessa conhecimento que não viu no treino — sem retreinar.

In [ ]:
docs = [
    "A política de troca permite devolução em até 30 dias com nota fiscal.",
    "O frete é grátis para compras acima de R$ 250 na região Sudeste.",
    "O horário de atendimento do suporte é de segunda a sexta, das 8h às 18h.",
    "Produtos em promoção não são elegíveis para troca, apenas para reembolso.",
    "A garantia padrão dos eletrônicos é de 12 meses contra defeitos de fabricação.",
]
idx_docs = faiss.IndexFlatIP(embed(docs[:1]).shape[1])
idx_docs.add(embed(docs))

def rag(pergunta, k=3):
    _, ids = idx_docs.search(embed([pergunta]), k=k)          # RECUPERAR
    trechos = [docs[int(i)] for i in ids[0]]
    contexto = "\n".join(f"- {t}" for t in trechos)
    prompt = (
        "Use SOMENTE o contexto para responder. Se não houver base, diga que não sabe.\n"
        f"Contexto:\n{contexto}\n\nPergunta: {pergunta}\nResposta:"
    )
    return llm.invoke(prompt).content, trechos                 # GERAR (Claude)

for pergunta in ["Posso trocar um produto comprado há 10 dias?",
                 "Qual o prazo da garantia dos eletrônicos?"]:
    resposta, trechos = rag(pergunta)
    print(f"Pergunta: {pergunta}")
    print(f"  Trecho-chave: {trechos[0]}")
    print(f"  Resposta: {resposta}\n")

## 4. Tool use — function calling chamando uma API externa

Agora o **próprio Claude decide** chamar uma ferramenta. Declaramos `cotacao_moeda` com `@tool`,
vinculamos ao modelo com `bind_tools` e rodamos o **loop de decisão**: o modelo emite `tool_calls`,
nós executamos (chamada HTTP real, com `timeout`) e devolvemos a observação até ele responder.
Toda chamada externa é um ponto de falha — por isso `timeout` e tratamento de erro.

In [ ]:
import os
import requests
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

AWESOME_API_KEY = os.getenv("AWESOME_API_KEY")  # token criado em awesomeapi.com.br (no .env)

@tool
def cotacao_moeda(par: str) -> str:
    """Consulta a cotação atual de um par de moedas. Ex.: par='USD-BRL' ou 'EUR-BRL'."""
    try:
        url = f"https://economia.awesomeapi.com.br/json/last/{par}"
        params = {"token": AWESOME_API_KEY} if AWESOME_API_KEY else {}
        dados = requests.get(url, params=params, timeout=10).json()   # chamada HTTP real
        chave = par.replace("-", "")
        if chave not in dados:  # API devolveu erro (token ausente/inválido, limite, par inexistente)
            return f"Erro da API ao consultar {par}: {dados}"
        return f"{par} = {dados[chave]['bid']}"
    except Exception as exc:
        return f"Erro ao consultar {par}: {type(exc).__name__}: {exc}"

ferramentas = [cotacao_moeda]
registro = {f.name: f for f in ferramentas}
llm_tools = llm.bind_tools(ferramentas)

mensagens = [HumanMessage("Qual a cotação do dólar e do euro em relação ao real hoje?")]
resposta = llm_tools.invoke(mensagens)
mensagens.append(resposta)

while resposta.tool_calls:                                     # LOOP DE DECISÃO
    for tc in resposta.tool_calls:
        obs = registro[tc["name"]].invoke(tc["args"])
        print(f"  [ferramenta] {tc['name']}({tc['args']}) -> {obs}")
        mensagens.append(ToolMessage(obs, tool_call_id=tc["id"]))
    resposta = llm_tools.invoke(mensagens)
    mensagens.append(resposta)

print("\nResposta final:", resposta.content)

## Conclusão

Com **embeddings + busca vetorial (RAG)** o agente recupera conhecimento atualizado; com **tool
use** (function calling) ele **age** no mundo e busca dados ao vivo — superando alucinação,
conhecimento estático e esquecimento. Lembre-se: RAG **mitiga**, não elimina, alucinações; e toda
API externa exige `timeout`, validação e tratamento de erro.